## Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import os
from glob import glob
from obspy import read, UTCDateTime as utc

## Write P/S waveforms to 3 component headers

In [2]:
eventdir_path = "/home/msseo/works/10.Earthquake_cycle_project/Gwangyang_sequence/manual_picks/"
eventdir_list = sorted(glob(os.path.join(eventdir_path, "20*")))

P_HEADER = 'a'
S_HEADER = 't0'

for eventdir in eventdir_list:
    sacfile_list = sorted(glob(os.path.join(eventdir, "*.sac")))
    
    # 성분을 제외한 공통 관측소 Prefix 추출 (예: .../20210720161418.KS.BAU)
    prefixes = set([f.rsplit('.', 2)[0] for f in sacfile_list])
    
    for prefix in prefixes:
        st_files = glob(f"{prefix}.*.sac")
        traces = {f: read(f)[0] for f in st_files}
        p_pick, s_pick = None, None
        
        # 1. 존재하는 P파 및 S파 Pick 시간 추출
        for tr in traces.values():
            val_p = tr.stats.sac.get(P_HEADER)
            val_s = tr.stats.sac.get(S_HEADER)
            if val_p is not None and val_p != -12345.0: p_pick = val_p
            if val_s is not None and val_s != -12345.0: s_pick = val_s
                
        # 2. Pick 정보가 하나라도 존재하면 3성분 모두에 적용 및 덮어쓰기
        if p_pick is not None or s_pick is not None:
            for f, tr in traces.items():
                if p_pick is not None: tr.stats.sac[P_HEADER] = p_pick
                if s_pick is not None: tr.stats.sac[S_HEADER] = s_pick
                tr.write(f, format="SAC")